# 零基准节点功率：预算对照与完整切割回放

直接优化 $p=(p_1,p_2,p_3)\ge0$，单位 kW。配变有功上限为 **142.5 kW**；图1显示 **0–140 kW** 的最低预算前沿，图2显示完整物理范围内的三维负荷域。

五个代码块依次为模块与参数、MP1、MP2、SP、持续迭代与回放。MP1/MP2共用SP产生的割：

$$\pi^T(h+Tx+Dp)\ge0,\qquad \pi\ge0,\quad W^T\pi=0,\quad r^T\pi\le1.$$

图2四个面板分别采用 **20,000 / 30,000 / 40,000 元 / 无建设预算上限**，使用同一坐标范围、同一割池。预算增加时，合法建设方案集合扩大，三维负荷域按包含关系扩大；最大总负荷不一定每次增加。

第5块输出可旋转的交互回放。可以切换“本次运行”和“已保存的完整实验”，按轮次或仅新增割播放；拖动三维图旋转，勾选“四图联动旋转”保持同一视角。切换轮次不会重置视角。[独立离线回放](notebook_results/node_power/complete/playback.html)使用同一界面。

中途显示的是外近似。只有所需建设方案的所有剩余顶点均通过SP后，才标为完整可行域；可行认证轮次仍保留运行变量与对偶，但不新增割。

**代码块1：载入模块、参数和持续状态。** `None` 表示取消建设预算上限，仍保留线路容量、电压和配变约束。

In [1]:
from pathlib import Path
from time import sleep
import numpy as np
import pandas as pd
import gurobipy as gp
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML

import planning_domain_demo as demo
from node_power_view import (
    PowerProcess, construction_table, state_table, dual_table,
    show_process, save_records,
)
from node_power_replay import display_replay

TARGET_POWER = np.array([40.0, 35.0, 25.0])  # MP1的节点功率目标；基准功率仍为零。
MP2_BUDGET = 30000.0                        # 代码块3的单次MP2查询预算。
BUDGETS = (20000.0, 30000.0, 40000.0, None) # 图2按建设预算比较四个完整负荷域。
FRONTIER_LIMIT = 140.0                     # 图1横轴上限；图2使用物理上限142.5 kW。
OUTPUT = Path("notebook_results/node_power/live")
COMPLETE_REPLAY = Path("notebook_results/node_power/complete/replay.json") # 已保存的完整实验；设None只看本次。
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

model = demo.build_model()                 # 写出电气行，提取W、h、T、D、r。
process = PowerProcess(model, TARGET_POWER, BUDGETS, FRONTIER_LIMIT)
print(f"SP: W={model.W.shape}, D={model.D.shape}；物理有功上限 {model.config.power_limit:g} kW。")
print("空割池已初始化；重跑块5继续，重跑块1重新开始。完整回放不会向本次求解注入未来割。")

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2685996


Academic license 2685996 - for non-commercial use only - registered to 20___@mail.scut.edu.cn


SP: W=(53, 8), D=(53, 3)；物理有功上限 142.5 kW。
空割池已初始化；重跑块5继续，重跑块1重新开始。完整回放不会向本次求解注入未来割。


**代码块 2：MP1 的单次构建与求解。**

$\min c^Tx$，满足 $x\in X_{\mathrm{tree}}$、$p=\bar p$、$p\ge0$、$\sum p_i\le142.5$ 及当前割。

这里使用空割池，得到的是待 SP 检查的候选方案，费用只是外近似下界。

In [2]:
mp1, x1, p1 = demo.build_master(model, "min_cost", TARGET_POWER, cuts=[])
mp1.optimize()                             # 只求一次 MP1，完整切割迭代见代码块5。
mp1_x = np.rint([v.X for v in x1.values()])
mp1_p = np.array([v.X for v in p1.values()])
display(construction_table(model, mp1_x))
display(pd.DataFrame({"node": [1, 2, 3], "p_kw": mp1_p}))
print(f"MP1 候选建设费：{model.cost @ mp1_x:,.0f} 元；尚未通过 SP。")
mp1.dispose()

,corridor,selected,line,cost_cny
0,01,True,L,0.0
1,12,True,L,0.0
2,13,True,L,0.0
3,02,False,—,0.0
4,23,False,—,0.0


,node,p_kw
0,1,40.0
1,2,35.0
2,3,25.0


MP1 候选建设费：0 元；尚未通过 SP。


**代码块 3：MP2 的单次构建与求解。**

$\max\sum_i p_i$，满足 $x\in X_{\mathrm{tree}}$、$c^Tx\le B$、$p\ge0$、$\sum p_i\le142.5$ 及当前割。空割池的目标值只是电气承载能力的上界。

In [3]:
mp2, x2, p2 = demo.build_master(model, "max_load", MP2_BUDGET, cuts=[])
mp2.optimize()
mp2_x = np.rint([v.X for v in x2.values()])
mp2_p = np.array([v.X for v in p2.values()])
display(construction_table(model, mp2_x))
display(pd.DataFrame({"node": [1, 2, 3], "p_kw": mp2_p}))
print(f"MP2 无割外上界：{mp2_p.sum():.6f} kW；不代表该负荷已电气可行。")
mp2.dispose()

,corridor,selected,line,cost_cny
0,01,True,L,0.0
1,12,True,L,0.0
2,13,True,L,0.0
3,02,False,—,0.0
4,23,False,—,0.0


,node,p_kw
0,1,142.5
1,2,0.0
2,3,0.0


MP2 无割外上界：142.500000 kW；不代表该负荷已电气可行。


**代码块 4：SP（SB）的单次调用。**

固定代码块2的 $(\bar x,\bar p)$，求 $\min\eta$，满足 $Ww\le h+T\bar x+D\bar p+r\eta$、$\eta\ge0$。

功率行尺度固定为 142.5，电压行尺度固定为 0.1351；两者均独立于零基准负荷。完整 53 行约束的原始残差为 `Ww-rhs`，松弛后余量为 `r*eta-(Ww-rhs)`。

In [4]:
sp = demo.feasibility_oracle(model, mp1_x, mp1_p)
display(state_table(model, sp))              # P_e、v_i、η；η>0 时为松弛解。
display(dual_table(model, sp))               # 每一个 π_j 对应的电气行名称及残差。
print(f"η*={sp.violation:.9g}；生成点割左端={sp.cut.constant + sp.cut.x_coeff @ mp1_x + sp.cut.load_coeff @ mp1_p:.9g}")
print("CUT: beta0 + beta_x @ x + beta_p @ p >= 0")
print("beta0 =", sp.cut.constant)
print("beta_x =", sp.cut.x_coeff)
print("beta_p =", sp.cut.load_coeff)           # 此块只演示查询；第5块仍从独立空割池开始。

,variable,value,unit
0,P_01,45.055966,kW
1,P_12,10.738271,kW
2,P_13,8.053703,kW
3,P_02,13.736008,kW
4,P_23,3.210288,kW
5,v_1,0.867275,p.u.^2
6,v_2,0.851877,p.u.^2
7,v_3,0.851877,p.u.^2
8,eta,0.096393,dimensionless


,row,constraint,SP inequality,pi=-Pi,rhs,Ww-rhs,relaxation=r*eta,relaxed_slack
0,E01,capacity_01_1,1 P_01 <= 35 + 142.5 eta,-0.000000,35.000000,10.055966,13.736008,3.680042e+00
1,E02,capacity_01_-1,-1 P_01 <= 35 + 142.5 eta,-0.000000,35.000000,-80.055966,13.736008,9.379197e+01
2,E03,capacity_12_1,1 P_12 <= 35 + 142.5 eta,-0.000000,35.000000,-24.261729,13.736008,3.799774e+01
3,E04,capacity_12_-1,-1 P_12 <= 35 + 142.5 eta,-0.000000,35.000000,-45.738271,13.736008,5.947428e+01
4,E05,capacity_13_1,1 P_13 <= 35 + 142.5 eta,-0.000000,35.000000,-26.946297,13.736008,4.068230e+01
5,E06,capacity_13_-1,-1 P_13 <= 35 + 142.5 eta,-0.000000,35.000000,-43.053703,13.736008,5.678971e+01
6,E07,capacity_02_1,1 P_02 <= 0 + 142.5 eta,0.001639,0.000000,13.736008,13.736008,0.000000e+00
7,E08,capacity_02_-1,-1 P_02 <= 0 + 142.5 eta,-0.000000,0.000000,-13.736008,13.736008,2.747202e+01
8,E09,capacity_23_1,1 P_23 <= 0 + 142.5 eta,-0.000000,0.000000,3.210288,13.736008,1.052572e+01
9,E10,capacity_23_-1,-1 P_23 <= 0 + 142.5 eta,-0.000000,0.000000,-3.210288,13.736008,1.694630e+01


η*=0.0963930416；生成点割左端=-0.0963930416
CUT: beta0 + beta_x @ x + beta_p @ p >= 0
beta0 = 0.3561358366365077
beta_x = [-0.15836709  0.          0.         -0.0788891   0.          0.
 -0.072224    0.          0.          0.05737906  0.10656111  0.16394018
  0.          0.          0.        ]
beta_p = [-0.00111711 -0.0016394  -0.0016394 ]


**代码块5：持续迭代、完整回放与手动旋转。**

每次 `process.step()` 执行一轮真实MP→SP。仍默认 `STEPS=10`、`PAUSE=1`；重跑此块会继续，完成后自动停止。计算结束后在同一输出区出现交互回放：

- **本次运行**：只含当前已经求出的轮次；重跑代码块5后更新。
- **已保存的完整实验**：直接回放随Notebook交付的完整默认实验，无需等本次逐步求完。
- **全部轮次 / 仅新增割**：前者保留所有顶点认证，后者跳过没有新割的轮次；均可拖动、前进、后退、播放、暂停和跳到最后。
- 三维图拖动旋转、滚轮缩放，可联动四个预算面板；回放中保持视角。各轮的建设、SP、全部53行对偶、累计割及范围在图下展开查看。

图1是给定总负荷时允许节点重新分配的最低预算。图2是给定预算时所有合法建设方案的非凸负荷域并集，四个面板均使用完整142.5 kW物理范围。灰线标最近加割前的边界，橙色标最近割对应的边界；不会把不同建设方案取整体凸包。

范围 `g_min/g_max` 固定当前轮的建设方案，在 $p\ge0,\sum_i p_i\le142.5$ 上计算。改变预算只改变纳入并集的建设方案，不改变SP电气模型或割的有效性。

Notebook需使用**受信任的输出**以运行嵌入的离线交互图；亦可直接打开上方独立回放。

In [5]:
STEPS = 10              # 1：每次运行只推进一轮；改成20自动推进20轮，完成后自动停止。
PAUSE = 1               # 自动推进时，两轮之间暂停的秒数。

for step in range(STEPS):
    if process.finished:
        break
    record = process.step()                    # 追加本轮，保留全部历史与割。
    clear_output(wait=True)
    show_process(process, record)              # 计算时逐轮展示建设、SP、对偶、割与预算对照图。
    if not process.finished and step + 1 < STEPS:
        sleep(PAUSE)

save_records(process, OUTPUT)
clear_output(wait=True)
display_replay(process, saved_replay=COMPLETE_REPLAY) # 同一输出区可切换本次进度/完整实验，并手动旋转。
print(f"本次运行：{len(process.history)}轮、{len(process.cuts)}条割；完成状态：{process.finished}。重跑本块继续。")

本次运行：10轮、10条割；完成状态：False。重跑本块继续。
